# Segment-based Inference: KDE Likelihood Multiplication vs Full Joint MCMC

Infer shared quadratic-in-time parameters `(a, b, c)` from many low-SNR segments.  
Compare approximate "multiply per-segment likelihoods via KDE" against full-joint MCMC.

In [1]:
import numpy as np
import emcee
from scipy.stats import gaussian_kde, norm
import corner
import matplotlib.pyplot as plt
from multiprocessing import Pool
import warnings
warnings.filterwarnings("ignore")

# ── Global settings ──
N_SEGMENTS = 20
N_POINTS = 50          # per segment
T_TOTAL = 1.0
SIGMA = 0.3
THETA_TRUE = np.array([0.5, -1.0, 0.8])
PRIOR_STD = 10.0
LABELS = ["a", "b", "c"]
NDIM = 3

In [2]:
# ── Model and probabilities ──

def model(theta, t_global, T_total):
    """f(t) = a + b*u + c*u^2, u = t_global / T_total"""
    a, b, c = theta
    u = t_global / T_total
    return a + b * u + c * u**2

def log_prior(theta):
    """Independent broad Gaussian priors, std=10."""
    return -0.5 * np.sum((theta / PRIOR_STD)**2)

def log_like_segment(theta, segment, T_total):
    """Gaussian log-likelihood for one segment."""
    resid = segment["y"] - model(theta, segment["t_global"], T_total)
    return -0.5 * np.sum((resid / segment["sigma"])**2)

def log_like_full(theta, segments, T_total):
    """Sum of log-likelihoods over all segments."""
    return sum(log_like_segment(theta, s, T_total) for s in segments)

def log_post_segment(theta, segment, T_total):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_like_segment(theta, segment, T_total)

def log_post_full(theta, segments, T_total):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_like_full(theta, segments, T_total)

In [3]:
# ── Simulate data ──

def simulate_data(N_segments, n_points, T_total, sigma, theta_true, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    Delta = T_total / N_segments
    segments = []
    for i in range(N_segments):
        t_local = rng.uniform(0, Delta, size=n_points)
        t_global = i * Delta + t_local
        y = model(theta_true, t_global, T_total) + rng.normal(0, sigma, size=n_points)
        segments.append(dict(i=i, t_local=t_local, t_global=t_global, y=y, sigma=sigma))
    return segments

segments = simulate_data(N_SEGMENTS, N_POINTS, T_TOTAL, SIGMA, THETA_TRUE)
print(f"Simulated {len(segments)} segments, {N_POINTS} points each, sigma={SIGMA}")

Simulated 20 segments, 50 points each, sigma=0.3


In [4]:
# ── Per-segment MCMC ──

def run_mcmc_segment(segment, T_total, nwalkers=24, nsteps=800, burn=400):
    """Run emcee on a single segment."""
    # Initialize near a broad ball around 0
    p0 = np.random.randn(nwalkers, NDIM) * 0.5
    sampler = emcee.EnsembleSampler(nwalkers, NDIM, log_post_segment, args=(segment, T_total))
    sampler.run_mcmc(p0, nsteps, progress=False)
    return sampler.get_chain(flat=True, discard=burn)

# Run all segments (this is the slow part)
print("Running per-segment MCMC...")
segment_samples = [run_mcmc_segment(seg, T_TOTAL) for seg in segments]
print(f"Done. Each segment: {segment_samples[0].shape[0]} samples")

Running per-segment MCMC...
Done. Each segment: 9600 samples


In [5]:
# ── KDE fits and approximate combination ──

def fit_kde(samples):
    """Fit scipy gaussian_kde to posterior samples."""
    return gaussian_kde(samples.T)

# Fit KDEs to each segment posterior
kdes = [fit_kde(s) for s in segment_samples]

def log_post_approx(theta, kdes):
    """
    Approximate combined posterior:
    log p_approx(theta) = log_prior(theta) + sum_i [log kde_i(theta) - log_prior(theta)]
    
    This subtracts the prior from each KDE (to get approx likelihood) then adds one prior back.
    Simplifies to: sum_i log kde_i(theta) - (N-1)*log_prior(theta)
    """
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    theta_col = theta.reshape(-1, 1)
    log_kdes = sum(np.log(kde(theta_col) + 1e-300)[0] for kde in kdes)
    return log_kdes - (len(kdes) - 1) * lp

In [6]:
# ── Full joint MCMC (gold standard) ──

def run_mcmc_full(segments, T_total, nwalkers=32, nsteps=2000, burn=800):
    p0 = np.random.randn(nwalkers, NDIM) * 0.1
    sampler = emcee.EnsembleSampler(nwalkers, NDIM, log_post_full, args=(segments, T_total))
    print("Running full joint MCMC...")
    sampler.run_mcmc(p0, nsteps, progress=True)
    print("Done.")
    return sampler.get_chain(flat=True, discard=burn)

samples_full = run_mcmc_full(segments, T_TOTAL)

Running full joint MCMC...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:07<00:00, 257.33it/s]

Done.


In [ ]:
# ── Approximate MCMC (sampling from KDE product) ──

def run_mcmc_approx(kdes, nwalkers=32, nsteps=3000, burn=1000):
    # Start near the mean of the full posterior (use mean of first KDE as rough guess)
    mean0 = segment_samples[0].mean(axis=0)
    p0 = mean0 + np.random.randn(nwalkers, NDIM) * 0.1
    sampler = emcee.EnsembleSampler(nwalkers, NDIM, log_post_approx, args=(kdes,))
    print("Running approximate (KDE product) MCMC...")
    sampler.run_mcmc(p0, nsteps, progress=True)
    print("Done.")
    return sampler.get_chain(flat=True, discard=burn)

samples_approx = run_mcmc_approx(kdes)

Running approximate (KDE product) MCMC...


 43%|█████████████████████████████████████████████▍                                                           | 1298/3000 [01:43<02:16, 12.50it/s]

In [ ]:
# ── Compare results ──

def print_comparison(samples_full, samples_approx, theta_true):
    print(f"{'':20s} {'a':>10s} {'b':>10s} {'c':>10s}")
    print(f"{'True':20s} {theta_true[0]:10.4f} {theta_true[1]:10.4f} {theta_true[2]:10.4f}")
    print(f"{'Full MCMC mean':20s} {samples_full.mean(0)[0]:10.4f} {samples_full.mean(0)[1]:10.4f} {samples_full.mean(0)[2]:10.4f}")
    print(f"{'Approx MCMC mean':20s} {samples_approx.mean(0)[0]:10.4f} {samples_approx.mean(0)[1]:10.4f} {samples_approx.mean(0)[2]:10.4f}")
    
    print("\n── Covariance (Full) ──")
    cov_full = np.cov(samples_full.T)
    print(np.array2string(cov_full, precision=5))
    
    print("\n── Covariance (Approx) ──")
    cov_approx = np.cov(samples_approx.T)
    print(np.array2string(cov_approx, precision=5))
    
    print("\n── Correlation (Full) ──")
    print(np.array2string(np.corrcoef(samples_full.T), precision=4))
    
    print("\n── Correlation (Approx) ──")
    print(np.array2string(np.corrcoef(samples_approx.T), precision=4))

print_comparison(samples_full, samples_approx, THETA_TRUE)

In [ ]:
# ── Model curves vs data ──

t_all = np.concatenate([s["t_global"] for s in segments])
y_all = np.concatenate([s["y"] for s in segments])
sort_idx = np.argsort(t_all)

t_plot = np.linspace(0, T_TOTAL, 300)
theta_full = samples_full.mean(axis=0)
theta_approx = samples_approx.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(t_all[sort_idx], y_all[sort_idx], s=4, alpha=0.3, color="gray", label="Data")
ax.plot(t_plot, model(THETA_TRUE, t_plot, T_TOTAL), "k-", lw=2, label="Truth")
ax.plot(t_plot, model(theta_full, t_plot, T_TOTAL), "C0--", lw=2,
        label=f"Full MCMC ({theta_full[0]:.2f}, {theta_full[1]:.2f}, {theta_full[2]:.2f})")
ax.plot(t_plot, model(theta_approx, t_plot, T_TOTAL), "C1:", lw=2.5,
        label=f"KDE approx ({theta_approx[0]:.2f}, {theta_approx[1]:.2f}, {theta_approx[2]:.2f})")

# Draw posterior uncertainty bands (2-sigma) from full MCMC
n_draw = 200
idx = np.random.choice(len(samples_full), n_draw, replace=False)
curves = np.array([model(samples_full[i], t_plot, T_TOTAL) for i in idx])
lo, hi = np.percentile(curves, [2.5, 97.5], axis=0)
ax.fill_between(t_plot, lo, hi, color="C0", alpha=0.15, label="Full MCMC 95% band")

ax.set_xlabel("$t_{\\rm global}$", fontsize=13)
ax.set_ylabel("$y$", fontsize=13)
ax.legend(fontsize=10)
ax.set_title("Model fits vs truth", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Corner plot ──

def make_corner(samples_full, samples_approx, theta_true):
    fig = corner.corner(
        samples_full, labels=LABELS, color="C0", 
        truths=theta_true, truth_color="k",
        hist_kwargs=dict(density=True),
        plot_datapoints=False, plot_density=False,
        levels=(0.68, 0.95),
        fill_contours=True, 
    )
    corner.corner(
        samples_approx, labels=LABELS, color="C1",
        fig=fig,
        hist_kwargs=dict(density=True),
        plot_datapoints=False, plot_density=False,
        levels=(0.68, 0.95),
        fill_contours=True,
    )
    # Legend
    from matplotlib.lines import Line2D
    handles = [
        Line2D([], [], color="C0", lw=2, label="Full joint MCMC"),
        Line2D([], [], color="C1", lw=2, label="KDE approx"),
        Line2D([], [], color="k", lw=1, ls="--", label="Truth"),
    ]
    fig.legend(handles=handles, loc="upper right", fontsize=12, frameon=True)
    fig.suptitle("Full Joint vs KDE-Approximate Posterior", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

make_corner(samples_full, samples_approx, THETA_TRUE)